In [ ]:
#standard
from dotenv import load_dotenv
load_dotenv()
import os
import sys
sys.path.append(os.getenv('PYTHONPATH')) 
import warnings
warnings.filterwarnings('ignore')
import pickle

#third party
import numpy as np
import hcp_utils as hcp
import matplotlib.pyplot as plt
from nilearn import plotting

#local
from src.utils.helpers import ComputeNoiseceiling

In [ ]:
#setup paths
datasets_root = os.path.join(os.getenv("DATASETS_ROOT", "/default/path/to/datasets")) #use default if DATASETS_ROOT env variable is not set.
dataset_root = os.path.join(datasets_root, "NaturalScenesDataset")
meta_dataset_root = os.path.join(datasets_root, "MOSAIC")
project_root = os.getenv("PROJECT_ROOT", "/default/path/to/datasets")
working_path = os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output", "ncsnr_mosaic")
if not os.path.exists(working_path):
    os.makedirs(working_path)
print(f"dataset_root: {dataset_root}")
print(f"project_root: {project_root}")


In [ ]:
for subject in range(1,9):
    print(f'starting sub-{subject:02}')
    with open(os.path.join(dataset_root, "derivatives", "GLM", f"sub-{subject:02}", "prepared_betas", f"sub-{subject:02}_organized_betas_task-train_normalized.pkl"), 'rb') as f: 
        betas_train, stimorder_train = pickle.load(f) #betas is shape numstim, numreps, numvertices
    with open(os.path.join(dataset_root, "derivatives", "GLM", f"sub-{subject:02}", "prepared_betas", f"sub-{subject:02}_organized_betas_task-test_normalized.pkl"), 'rb') as f: 
        betas_test, stimorder_test = pickle.load(f) #betas is shape numstim, numreps, numvertices
    betas = np.vstack((betas_train, betas_test)).T
    print(betas.shape)

    print("calculating ncsnr...")
    ncsnr, _= ComputeNoiseceiling(betas, n='avg').compute_noiseceiling() #for ncsnr, the n does not matter.
    del betas
    save_path = os.path.join(working_path, f"sub-{subject:02}")
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    lh_ncsnr = hcp.left_cortex_data(ncsnr)
    rh_ncsnr = hcp.right_cortex_data(ncsnr)
    print(f"lh ncsnr shape: {lh_ncsnr.shape}")
    print(f"rh ncsnr shape: {rh_ncsnr.shape}")

    np.save(os.path.join(save_path, f'lh.ncsnr_mosaic.npy'), lh_ncsnr)
    np.save(os.path.join(save_path, f'rh.ncsnr_mosaic.npy'), rh_ncsnr)
    del lh_ncsnr, rh_ncsnr, ncsnr
print("done saving mosaic ncsnr")

In [ ]:
def plot_flatmap(stat, cmap='hot', subject=None, save_flag=False):
    cortex_data_left = hcp.left_cortex_data(stat)
    cortex_data_right = hcp.right_cortex_data(stat)

    #determine global min/max for consistent color scaling
    datamin = min(np.nanmin(cortex_data_left), np.nanmin(cortex_data_right))
    datamax = max(np.nanmax(cortex_data_left), np.nanmax(cortex_data_right))
    threshold = None
    vmin=datamin
    vmax=datamax
    #create a figure with multiple axes to plot each anatomical image
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4), subplot_kw={'projection': '3d'})
    plt.subplots_adjust(wspace=0)
    im = plotting.plot_surf(hcp.mesh.flat_left, cortex_data_left,
            threshold=threshold, bg_map=hcp.mesh.sulc_left, 
            colorbar=False, cmap=cmap, 
            vmin=vmin, vmax=vmax,
            axes = axes[0])
    im = plotting.plot_surf(hcp.mesh.flat_right, cortex_data_right,
            threshold=threshold, bg_map=hcp.mesh.sulc_right, 
            colorbar=False, cmap=cmap, 
            vmin=vmin, vmax=vmax,
            axes = axes[1])
    
    #flip along the horizontal
    axes[0].invert_yaxis()
    axes[1].invert_yaxis()

    #create colorbar
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.6)

    cbar.set_ticks([0, round(datamax,2)])
    cbar.set_ticklabels([0, round(datamax,2)])
    if save_flag:
        plt.savefig(os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output", "plots", f"{subject}_fsLR32k_mosaic.png"), dpi=300)
    plt.show()

In [ ]:
for subject in range(1,9):
    print(f'starting sub-{subject:02}')
    save_path = os.path.join(working_path, f"sub-{subject:02}")

    lh_ncsnr = np.load(os.path.join(save_path, f'lh.ncsnr_mosaic.npy'))
    rh_ncsnr = np.load(os.path.join(save_path, f'rh.ncsnr_mosaic.npy'))

    vertex_info = hcp.vertex_info
    stat = np.hstack((lh_ncsnr[vertex_info.grayl],rh_ncsnr[vertex_info.grayr]))
    plot_flatmap(stat, subject=f"sub-{subject:02}", save_flag=True)
print("done saving mosaic ncsnr")